# Capstone 1 — Retail Sales & Inventory Analytics
### Microsoft Fabric Medallion Capstone

**Business scenario:** A mid-size retail chain ("UrbanMart") wants a single analytics platform covering sales performance and stock-out risk across its stores.

**Fabric capability highlighted:** Lakehouse Medallion architecture + a Gold layer designed for Power BI **Direct Lake** consumption.

**What this notebook builds, end to end, in one place:**
1. Synthetic source data (stores, products, sales, inventory snapshots)
2. Bronze — raw append-only ingestion
3. Silver — cleansed, conformed, deduplicated
4. Gold — star schema + two business marts: `mart_sales_performance` and `mart_stockout_risk`

Run all cells top to bottom. Attach this notebook to a Lakehouse (e.g. `retail_capstone_lakehouse`) before running.

In [ ]:
%pip install faker --quiet

In [ ]:
import random
from datetime import datetime, timedelta
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from faker import Faker

fake = Faker()
Faker.seed(7)
random.seed(7)

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

NUM_STORES = 20
NUM_PRODUCTS = 300
NUM_DAYS = 120
AVG_SALES_PER_DAY = 800

## 1. Generate source data (stores, products, sales, inventory)

In [ ]:
categories = ["Apparel", "Footwear", "Electronics", "Home & Kitchen", "Groceries", "Beauty"]
regions = ["North", "South", "East", "West"]

stores = [{
    "store_id": f"ST{i:03d}",
    "store_name": f"{fake.city()} Store",
    "region": random.choice(regions),
    "sqft": random.choice([2000, 3500, 5000, 8000]),
} for i in range(1, NUM_STORES + 1)]

products = [{
    "product_id": f"PRD{i:04d}",
    "product_name": fake.word().capitalize() + " " + random.choice(["Tee", "Jacket", "Blender", "Phone Case", "Lamp", "Shoes"]),
    "category": random.choice(categories),
    "unit_price": round(random.uniform(5, 500), 2),
    "unit_cost": None,  # filled below
} for i in range(1, NUM_PRODUCTS + 1)]
for p in products:
    p["unit_cost"] = round(p["unit_price"] * random.uniform(0.4, 0.7), 2)

df_stores = pd.DataFrame(stores)
df_products = pd.DataFrame(products)
# inject a couple of duplicate product rows to give Silver something to dedupe
df_products = pd.concat([df_products, df_products.sample(5, random_state=1)], ignore_index=True)

print(f"{len(df_stores)} stores, {len(df_products)} product rows (incl. duplicates)")

In [ ]:
store_ids = df_stores["store_id"].tolist()
product_ids = df_products["product_id"].unique().tolist()
start_date = datetime.utcnow() - timedelta(days=NUM_DAYS)

sales = []
sid = 1
for d in range(NUM_DAYS):
    day = start_date + timedelta(days=d)
    for _ in range(int(random.gauss(AVG_SALES_PER_DAY, 60))):
        sales.append({
            "sale_id": f"SALE{sid:08d}",
            "store_id": random.choice(store_ids),
            "product_id": random.choice(product_ids),
            "sale_date": day.date().isoformat(),
            "quantity": random.randint(1, 5),
            "discount_pct": random.choice([0, 0, 0, 5, 10, 20]),
        })
        sid += 1

df_sales = pd.DataFrame(sales)
print(f"Generated {len(df_sales):,} sales line items")

In [ ]:
# Weekly inventory snapshots per store/product (some negative/nonsensical values injected on purpose)
inventory = []
snapshot_dates = [start_date + timedelta(days=7 * w) for w in range(NUM_DAYS // 7)]
for snap_date in snapshot_dates:
    for store_id in store_ids:
        for product_id in random.sample(product_ids, k=min(50, len(product_ids))):
            qty = random.randint(-5, 200)  # negative values are intentional bad data
            inventory.append({
                "store_id": store_id,
                "product_id": product_id,
                "snapshot_date": snap_date.date().isoformat(),
                "quantity_on_hand": qty,
                "reorder_point": random.randint(10, 40),
            })

df_inventory = pd.DataFrame(inventory)
print(f"Generated {len(df_inventory):,} inventory snapshot rows "
      f"({(df_inventory['quantity_on_hand'] < 0).sum()} with negative qty — intentional bad data)")

## 2. Bronze layer — append-only raw ingestion

In [ ]:
def to_bronze(pdf, table_name):
    sdf = spark.createDataFrame(pdf).withColumn("_ingestion_timestamp", F.current_timestamp())
    sdf.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table_name)
    print(f"{table_name}: {sdf.count():,} rows")

to_bronze(df_stores, "bronze.stores")
to_bronze(df_products, "bronze.products")
to_bronze(df_sales, "bronze.sales")
to_bronze(df_inventory, "bronze.inventory_snapshots")

## 3. Silver layer — clean, dedupe, and apply data-quality rules

In [ ]:
silver_stores = spark.table("bronze.stores").dropDuplicates(["store_id"])
silver_stores.write.format("delta").mode("overwrite").saveAsTable("silver.stores")

w = Window.partitionBy("product_id").orderBy(F.col("_ingestion_timestamp").desc())
silver_products = (spark.table("bronze.products")
    .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
    .withColumn("unit_price", F.col("unit_price").cast("decimal(10,2)"))
    .withColumn("unit_cost", F.col("unit_cost").cast("decimal(10,2)")))
silver_products.write.format("delta").mode("overwrite").saveAsTable("silver.products")

silver_sales = (spark.table("bronze.sales")
    .withColumn("sale_date", F.to_date("sale_date"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .dropDuplicates(["sale_id"]))
silver_sales.write.format("delta").mode("overwrite").saveAsTable("silver.sales")

print(f"silver.stores: {silver_stores.count():,} | silver.products: {silver_products.count():,} "
      f"| silver.sales: {silver_sales.count():,}")

In [ ]:
# Inventory: quarantine negative quantities instead of silently dropping them
bronze_inv = spark.table("bronze.inventory_snapshots").withColumn("snapshot_date", F.to_date("snapshot_date"))

passed_inv = bronze_inv.filter("quantity_on_hand >= 0")
failed_inv = (bronze_inv.filter("quantity_on_hand < 0")
    .withColumn("_dq_reason", F.lit("negative quantity_on_hand")))

failed_inv.write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable("silver.inventory_quarantine")
passed_inv.write.format("delta").mode("overwrite").saveAsTable("silver.inventory_snapshots")

print(f"silver.inventory_snapshots: {passed_inv.count():,} passed | {failed_inv.count():,} quarantined")

## 4. Gold layer — star schema + business marts

In [ ]:
dim_store = spark.table("silver.stores")
dim_product = spark.table("silver.products")
dim_store.write.format("delta").mode("overwrite").saveAsTable("gold.dim_store")
dim_product.write.format("delta").mode("overwrite").saveAsTable("gold.dim_product")

fact_sales = (spark.table("silver.sales")
    .join(dim_product.select("product_id", "unit_price", "unit_cost", "category"), "product_id")
    .withColumn("gross_revenue", F.round(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct") / 100), 2))
    .withColumn("gross_margin", F.round(F.col("gross_revenue") - (F.col("quantity") * F.col("unit_cost")), 2)))
fact_sales.write.format("delta").mode("overwrite").saveAsTable("gold.fact_sales")

fact_inventory = spark.table("silver.inventory_snapshots")
fact_inventory.write.format("delta").mode("overwrite").saveAsTable("gold.fact_inventory")

print(f"gold.fact_sales: {fact_sales.count():,} | gold.fact_inventory: {fact_inventory.count():,}")

In [ ]:
mart_sales_performance = (fact_sales
    .groupBy("store_id", "category")
    .agg(F.sum("quantity").alias("units_sold"),
         F.sum("gross_revenue").alias("total_revenue"),
         F.sum("gross_margin").alias("total_margin"))
    .join(dim_store.select("store_id", "store_name", "region"), "store_id"))

mart_sales_performance.write.format("delta").mode("overwrite").saveAsTable("gold.mart_sales_performance")

mart_stockout_risk = (fact_inventory
    .withColumn("below_reorder_point", F.col("quantity_on_hand") < F.col("reorder_point"))
    .join(dim_product.select("product_id", "product_name", "category"), "product_id")
    .join(dim_store.select("store_id", "store_name"), "store_id")
    .filter("below_reorder_point = true")
    .select("store_id", "store_name", "product_id", "product_name", "category",
            "snapshot_date", "quantity_on_hand", "reorder_point"))

mart_stockout_risk.write.format("delta").mode("overwrite").saveAsTable("gold.mart_stockout_risk")

print(f"gold.mart_sales_performance: {mart_sales_performance.count():,} rows")
print(f"gold.mart_stockout_risk: {mart_stockout_risk.count():,} at-risk (store, product, week) rows")

## 5. Capstone checkpoint
Suggested Power BI report pages: **Sales Performance** (region/category revenue & margin trend), **Stock-out Risk** (a table over `mart_stockout_risk` with conditional formatting), and a **Store Scorecard** drill-through from `dim_store`.

In [ ]:
for t in ["gold.dim_store", "gold.dim_product", "gold.fact_sales", "gold.fact_inventory",
          "gold.mart_sales_performance", "gold.mart_stockout_risk"]:
    print(f"{t:35s} -> {spark.table(t).count():,} rows")
print("\nCapstone 1 (Retail Sales & Inventory Analytics) complete.")